# Initializing the environment

In [ ]:
from gymnasium import make
from samples.llm_interface import OGPT4Interfacer
import os
import random
from natural20.gym.tools import compute_available_moves
os.environ["OPENAI_API_KEY"] = ""

MAX_EPISODES = 5
n_player1 = 2
n_player2 = 2
# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
all_classes = ['halfling_rogue.yml', 'high_elf_fighter.yml']
all_players = ["Alysha", "Bernard", "Cedric", "Didier", "Eric", "Francois", "Gertrude", "Heloise", "Isabelle"]
players = random.sample(all_players, n_player1 + n_player2)
a_player = [(random.choice(all_classes), player) for player in players[:n_player1]]
e_player = [(random.choice(all_classes), player) for player in players[n_player1:]]
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=a_player,
    enemies=e_player,
    control_groups=["a","b"]
    )

envi = env.env.env
observation, info = env.reset(seed = random.randint(0,1000))




agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agents[character.name] = (OGPT4Interfacer(debug=False, explain=True, name=character.name), gr, character)

agents


Isabelle rolled initiative d20(18) + 5 value 23.2
Francois rolled initiative d20(18) + 5 value 23.2
Gertrude rolled initiative d20(9) + 5 value 14.2
Heloise rolled initiative d20(13) + 5 value 18.2
Isabelle rolled initiative d20(10) + 5 value 15.2
Francois rolled initiative d20(19) + 5 value 24.2
Gertrude rolled initiative d20(15) + 5 value 20.2
Heloise rolled initiative d20(6) + 5 value 11.2
Combat begins with 4 players.
Players: <p>Isabelle (fighter-2) Team a</p>
<p>Francois (rogue-2) Team a</p>
<p>Gertrude (rogue-2) Team b</p>
<p>Heloise (fighter-2) Team b</p>
======== Francois starts their turn. ========
======== Francois starts their turn. ========


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/spaces/box.py:423: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


{'Francois': (<samples.llm_interface.OGPT4Interfacer at 0x7f2cf1e2c980>,
  'a',
  Francois),
 'Gertrude': (<samples.llm_interface.OGPT4Interfacer at 0x7f2cf17de5d0>,
  'b',
  Gertrude),
 'Isabelle': (<samples.llm_interface.OGPT4Interfacer at 0x7f2cf16dee90>,
  'a',
  Isabelle),
 'Heloise': (<samples.llm_interface.OGPT4Interfacer at 0x7f2cf16f8050>,
  'b',
  Heloise)}

In [2]:
class Experiment():
    def __init__(self, environment, dnd_environment, agents, debug=False):
        self.env = environment
        self.dnd_environment = dnd_environment
        self.agents = agents
        self.debug = debug
        self.backlog = []
        self.conversations = []
    
    def get_obs_inf(self, player):
        p_observation = self.dnd_environment.generate_observation(player)
        p_available_moves = compute_available_moves(self.dnd_environment.session, self.dnd_environment.map, player, self.dnd_environment.battle, self.dnd_environment.weapon_mappings, self.dnd_environment.spell_mappings)
        p_info = self.dnd_environment._info(p_available_moves, player)
        return p_observation, p_info
    
    def update_all_agents(self, agents_name, sender, content):
        for name in agents_name:
            self.agents[name][0].register_conversation(sender, content)
        self.conversations[-1].append((sender, content))

    def initiate_conversation(self, agents_name):
        for name in agents_name:
            self.agents[name][0].initiate_conversation()
        self.conversations.append([])

    def close_conversation(self, agents_name):
        for name in agents_name:
            obs, inf = self.get_obs_inf(self.agents[name][2])
            self.agents[name][0].close_conversation(obs, inf, self.get_players_pos())
        self.conversations[-1].append((None, "Conversation closed"))

    def run_conversation(self, sender, content):
        sender_gr = self.agents[sender][1]
        agent_in_the_conv = []
        for name, (_, gr, _) in self.agents.items():
            if sender_gr == gr and name != sender:
                agent_in_the_conv.append(name)
        agent_in_the_conv.append(sender)
        self.initiate_conversation(agent_in_the_conv)
        self.update_all_agents(agent_in_the_conv, sender, content)
        conv_alive = True
        conv_step = 0
        while conv_alive:
            conv_step += 1
            conv_alive = False
            for name in agent_in_the_conv:
                obs, inf = self.get_obs_inf(self.agents[name][2])
                action, descrition,  content = self.agents[name][0].select_action_for_state(obs, inf, self.get_players_pos(), is_conversation=True)
                if action == -2:
                    self.update_all_agents(agent_in_the_conv, name, content)
                    conv_alive = True
                elif action != -3:
                    raise ValueError(f"A non conversation action {action} was used during a conversation by agent {name}")
        self.close_conversation(agent_in_the_conv)
    
    def step(self):
        current_agent, current_group, current_character = self.agents[self.dnd_environment.battle.current_turn().name]
        obs, inf = self.get_obs_inf(current_character)
        # Manual removala of help action since it crashes
        help_index = []
        for i, el in enumerate(inf["available_moves"]):
            if el[0] == 14:
                help_index.append(i)
        for index in help_index[::-1]:
            inf["available_moves"].pop(index)
        
        action, descrition, content = current_agent.select_action_for_state(obs, inf, self.get_players_pos())
        print(f"The chosen action is : {action}")
        self.backlog.append((current_character.name, action, descrition))
        if action != -1:
            _, _, terminal, _, _ = self.env.step(action)
        else :
            self.backlog.append((current_character.name, -1, len(self.conversations)))
            self.run_conversation(sender=current_character.name, content=content)
            terminal = False
        return terminal
    
    def get_players_pos(self):
        return {player: self.dnd_environment.battle.maps[0].entity_or_object_pos(player) for (_, _, player) in self.agents.values()}
    
    def run_till_end(self, max_step= 30):
        done = False
        step = 0
        while not done and step < max_step:
            if self.debug: print(f"\n\n________________________________________________________________________________\n Starting step {step}:\n")
            health = []
            for player, pos in self.get_players_pos().items():
                health.append(player.health_percent())
                print(f"Player {player.name} has {player.health_percent()}% life points, {pos}")
            self.backlog.append(health)
            done = self.step()
            step += 1

In [3]:
expe = Experiment(env, env.env.env, agents, debug=True)
expe.run_till_end(max_step=100)



________________________________________________________________________________
 Starting step 0:

Player Francois has 100% life points, [18, 0]
Player Gertrude has 100% life points, [11, 4]
Player Isabelle has 100% life points, [1, 2]
Player Heloise has 100% life points, [4, 4]
{'action_id': 7, 'description': 'hide action', 'explanation': "As a rogue, using the Hide action on the first turn is strategic. It may grant me advantage on my next attack (Sneak Attack) and helps avoid becoming an immediate target. I'll maneuver for a good vantage point next round if needed."}
The chosen action is : (15, (-1, -1), (0, 0), 0, 0)
Francois successfully hides with a d20(20) + 5=25 stealth.


________________________________________________________________________________
 Starting step 1:

Player Francois has 100% life points, [18, 0]
Player Gertrude has 100% life points, [11, 4]
Player Isabelle has 100% life points, [1, 2]
Player Heloise has 100% life points, [4, 4]


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(


{'action_id': 0, 'description': 'end my turn', 'explanation': "I already used my action to Hide this turn and only have bonus actions left, but as a rogue, none of my current bonus actions (Dash, Disengage) are helpful if I'm staying concealed and haven't revealed myself. I'll maintain my hidden position to potentially gain advantage and Sneak Attack next round."}
The chosen action is : (-1, (0, 0), (0, 0), 0, 0)
======== Gertrude starts their turn. ========
======== Gertrude starts their turn. ========


________________________________________________________________________________
 Starting step 2:

Player Francois has 100% life points, [18, 0]
Player Gertrude has 100% life points, [11, 4]
Player Isabelle has 100% life points, [1, 2]
Player Heloise has 100% life points, [4, 4]
{'action_id': 4, 'description': 'hide action', 'explanation': "As a rogue, taking the Hide action at the start of combat is a strong choice to potentially gain advantage on my next attack (for sneak attack) a

In [ ]:
expe.dnd_environment.battle.

In [ ]:
expe.dnd_environment.players

[('a', 'H', Cedric, [18, 6]),
 ('a', 'H', Alysha, [10, 3]),
 ('b', 'E', Bernard, [2, 5]),
 ('b', 'E', Isabelle, [12, 5])]

In [ ]:
expe.backlog

[[100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 ('Bernard', (15, (-1, -1), (0, 0), 0, 0), 'hide action'),
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 ('Bernard', (-1, (0, 0), (0, 0), 0, 0), 'end my turn'),
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 ('Alysha', (15, (-1, -1), (0, 0), 0, 0), 'hide action'),
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 ('Alysha', (1, (0, -1), (0, 0), 0, 0), 'move 5ft up'),
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 ('Alysha', (1, (-1, -1), (0, 0), 0, 0), 'move 5ft up and to the left'),
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 ('Alysha', (1, (-1, -1), (0, 0), 0, 0), 'move 5ft up and to the left'),
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [100, 100, 100, 100],
 [1

In [ ]:
from natural20.gym.llm_helpers.metrics import combat_metrics, combat_score


expe.backlog
metrics = combat_metrics(expe.dnd_environment)
print(metrics)
score = combat_score(metrics)
print(score)

{'win': False, 'turns_taken': 27, 'survivors': {'Bernard': (16, 16), 'Didier': (18, 24), 'Gertrude': (20, 24)}}
-1.1666666666666643


In [ ]:
env.env.env.players

[('a', 'H', Alysha, [10, 4]),
 ('a', 'H', Bernard, [1, 2]),
 ('b', 'E', Didier, [3, 6]),
 ('b', 'E', Gertrude, [9, 3])]

In [ ]:
agents["Didier"][0].summary
# agents

'Combat is underway. Bernard is still the primary threat, as he remains at full health, and I continue to prioritize him using my longbow from my current position, maintaining line of sight. This round, I have already used my action to attack him and see no advantage in repositioning or using other actions at this time, so I ended my turn to conserve resources. I remain alert to Bernard’s next move and the general flow of battle. Alysha is down, but I will continue to monitor Gertrude’s position and support as needed. The plan is to keep pressure on Bernard while staying flexible for any changes in the situation.'

In [ ]:
# Select an action based on the initial state
from natural20.gym.llm_helpers.metrics import combat_metrics, combat_score


current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]

action = current_agent.select_action_for_state(observation, info, env.env.env.players)
print(f"Selected action: {action}")
# terminal = False
# episode = 0
# while not terminal and episode < MAX_EPISODES:
#     episode += 1
#     observation, reward, terminal, truncated, info = env.step(action)
#     if not terminal and not truncated:
#         print(env.render())
#         current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]
#         action = current_agent.select_action_for_state(observation, info)
#         print(f"Selected action: {action}")

#     if terminal or truncated:
#         print(f"Reward: {reward}")
#         break
# action

{'action_id': 13, 'description': 'second wind action', 'explanation': "I'm currently at 0 HP and unconscious as Alysha, so I can't take any actions, including using Second Wind. Therefore, I am effectively unable to act until stabilized or healed."}
Selected action: ((8, (-1, -1), (0, 0), 0, 0), 'second wind action', None)


In [ ]:
info

{'available_moves': [(0, (0, 0), (7, -2), 13, 1),
  (0, (0, 0), (-2, -4), 13, 1),
  (15, (-1, -1), (0, 0), 0, 0),
  (4, (-1, -1), (0, 0), 0, 0),
  (2, (-1, -1), (0, 0), 0, 0),
  (3, (-1, -1), (0, 0), 0, 0),
  (1, (-1, -1), (0, 0), 0, 0),
  (1, (-1, 0), (0, 0), 0, 0),
  (1, (0, -1), (0, 0), 0, 0),
  (1, (1, -1), (0, 0), 0, 0),
  (1, (1, 0), (0, 0), 0, 0),
  (10, (-1, -1), (0, 0), 0, 0),
  (8, (-1, -1), (0, 0), 0, 0),
  (14, (-1, -1), (0, 0), 0, 0),
  (17, (-1, -1), (0, 0), 0, 0),
  (16, (-1, -1), (0, 0), 0, 0),
  (-1, (0, 0), (0, 0), 0, 0)],
 'current_index': 0,
 'group': 'b',
 'round': 0,
 'health': 24,
 'max_health': 24,
 'weapon_mappings': {'unarmed': 0,
  'battleaxe': 1,
  'dagger': 2,
  'quarterstaff': 3,
  'sling': 4,
  'dart': 5,
  'greatclub': 6,
  'hand_crossbow': 7,
  'handaxe': 8,
  'javelin': 9,
  'heavy_crossbow': 10,
  'light_crossbow': 11,
  'light_hammer': 12,
  'longbow': 13,
  'longsword': 14,
  'rapier': 15,
  'scimitar': 16,
  'shortsword': 17,
  'shortbow': 18,
  's

In [ ]:
observation.keys()

dict_keys(['map', 'turn_info', 'conditions', 'health_pct', 'player_equipped', 'ally_name', 'enemy_name', 'ally_reactions', 'health_ally', 'health_enemy', 'ally_conditions', 'enemy_conditions', 'enemy_reactions', 'player_ac', 'ally_ac', 'enemy_ac', 'ability_info', 'player_type', 'ally_type', 'enemy_type', 'spell_slots', 'movement', 'is_reaction'])

In [ ]:
observation["health_enemy"]

array([1., 1.])